# Анализ продуктовой воронки и когорт

В этом проекте я разбираю поведение пользователей в интернет-магазине по событийным данным Retailrocket. Данные представлены как последовательность действий пользователей: просмотр товара, добавление в корзину и покупка.

Основная задача проекта — посмотреть, как пользователи проходят по продуктовой воронке, на каком этапе теряется больше всего пользователей и как меняется возвращаемость по когортам. Такой анализ близок к задачам продуктового аналитика: сначала нужно аккуратно собрать метрики, а затем по ним сформулировать понятные продуктовые выводы.

## 1. Импорт библиотек

In [ ]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2. Загрузка данных

Для работы нужен файл:

```text
data/raw/events.csv
```

В датасете есть события интернет-магазина: просмотры товаров, добавления в корзину и покупки. Для каждого события известны пользователь, товар и время действия.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "events.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Файл не найден: {DATA_PATH}\n"
        "Скачайте датасет и положите файл events.csv в папку data/raw/.\n"
        "Команда: kaggle datasets download -d retailrocket/ecommerce-dataset -p data/raw --unzip"
    )

events = pd.read_csv(DATA_PATH)
events.head()

## 3. Подготовка данных

В исходном файле время хранится в миллисекундах. Я перевожу его в обычный формат даты и времени, а также добавляю несколько признаков, которые понадобятся дальше: день, неделя и месяц события.

In [ ]:
events.columns = events.columns.str.lower()

events["event_time"] = pd.to_datetime(events["timestamp"], unit="ms")
events["event_date"] = events["event_time"].dt.date
events["event_month"] = events["event_time"].dt.to_period("M").dt.to_timestamp()
events["event_week"] = events["event_time"].dt.to_period("W").dt.start_time
events["event_dayofweek"] = events["event_time"].dt.day_name()

events.info()

In [ ]:
events.head()

## 4. Первичный обзор датасета

Сначала проверяю общий размер данных: количество строк, число пользователей, число товаров и период наблюдения. После этого отдельно смотрю, сколько событий приходится на каждый тип действия.

In [ ]:
overview = pd.DataFrame({
    "metric": [
        "rows",
        "unique_visitors",
        "unique_items",
        "start_date",
        "end_date"
    ],
    "value": [
        len(events),
        events["visitorid"].nunique(),
        events["itemid"].nunique(),
        events["event_time"].min(),
        events["event_time"].max()
    ]
})

overview

In [ ]:
event_summary = (
    events
    .groupby("event")
    .agg(
        events_count=("event", "size"),
        unique_visitors=("visitorid", "nunique"),
        unique_items=("itemid", "nunique")
    )
    .sort_values("events_count", ascending=False)
)

event_summary

In [ ]:
ax = event_summary["events_count"].plot(kind="bar", figsize=(7, 4))
ax.set_title("Количество событий по типам")
ax.set_xlabel("Тип события")
ax.set_ylabel("Количество событий")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. SQL-анализ через DuckDB

Часть расчётов я делаю через SQL, потому что для продуктовой аналитики это основной инструмент работы с событийными таблицами. DuckDB удобен тем, что позволяет выполнять SQL-запросы прямо к датафрейму pandas.

In [ ]:
con = duckdb.connect()
con.register("events_df", events)

con.execute("""
CREATE OR REPLACE VIEW events AS
SELECT
    timestamp,
    visitorid,
    event,
    itemid,
    transactionid,
    event_time,
    event_month,
    event_week,
    event_dayofweek
FROM events_df
""")

In [ ]:
sql_event_summary = con.execute("""
SELECT
    event,
    COUNT(*) AS events_count,
    COUNT(DISTINCT visitorid) AS unique_visitors,
    COUNT(DISTINCT itemid) AS unique_items
FROM events
GROUP BY event
ORDER BY events_count DESC
""").df()

sql_event_summary

## 6. Продуктовая воронка

В этом проекте я рассматриваю базовую воронку интернет-магазина:

```text
view → addtocart → transaction
```

На первом шаге считаю воронку на уровне пользователей. Пользователь попадает на этап, если у него было хотя бы одно соответствующее событие. Такой подход даёт быстрый общий срез: сколько пользователей только смотрели товары, сколько дошли до корзины и сколько совершили покупку.

In [ ]:
visitor_flags = con.execute("""
WITH visitor_flags AS (
    SELECT
        visitorid,
        MAX(CASE WHEN event = 'view' THEN 1 ELSE 0 END) AS has_view,
        MAX(CASE WHEN event = 'addtocart' THEN 1 ELSE 0 END) AS has_addtocart,
        MAX(CASE WHEN event = 'transaction' THEN 1 ELSE 0 END) AS has_transaction
    FROM events
    GROUP BY visitorid
)
SELECT
    visitorid,
    has_view,
    has_addtocart,
    has_transaction
FROM visitor_flags
""").df()

visitor_flags.head()

In [ ]:
funnel = pd.DataFrame({
    "step": ["view", "addtocart", "transaction"],
    "users": [
        visitor_flags["has_view"].sum(),
        ((visitor_flags["has_view"] == 1) & (visitor_flags["has_addtocart"] == 1)).sum(),
        ((visitor_flags["has_view"] == 1) & (visitor_flags["has_transaction"] == 1)).sum()
    ]
})

funnel["conversion_from_previous_step"] = funnel["users"] / funnel["users"].shift(1)
funnel["conversion_from_first_step"] = funnel["users"] / funnel.loc[0, "users"]

funnel

In [ ]:
ax = funnel.set_index("step")["users"].plot(kind="bar", figsize=(7, 4))
ax.set_title("Пользовательская воронка")
ax.set_xlabel("Этап воронки")
ax.set_ylabel("Количество пользователей")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Строгая последовательная воронка

Обычная воронка выше учитывает сам факт события, но не проверяет порядок действий. Поэтому отдельно считаю более строгую версию: пользователь должен сначала посмотреть товар, затем добавить товар в корзину и только после этого совершить покупку.

Такой вариант ближе к реальному пользовательскому пути и помогает не завышать конверсию между этапами.

In [ ]:
first_events = con.execute("""
SELECT
    visitorid,
    MIN(CASE WHEN event = 'view' THEN event_time END) AS first_view_time,
    MIN(CASE WHEN event = 'addtocart' THEN event_time END) AS first_addtocart_time,
    MIN(CASE WHEN event = 'transaction' THEN event_time END) AS first_transaction_time
FROM events
GROUP BY visitorid
""").df()

first_events.head()

In [ ]:
strict_funnel = pd.DataFrame({
    "step": ["view", "addtocart_after_view", "transaction_after_cart"],
    "users": [
        first_events["first_view_time"].notna().sum(),
        (
            first_events["first_view_time"].notna()
            & first_events["first_addtocart_time"].notna()
            & (first_events["first_addtocart_time"] >= first_events["first_view_time"])
        ).sum(),
        (
            first_events["first_view_time"].notna()
            & first_events["first_addtocart_time"].notna()
            & first_events["first_transaction_time"].notna()
            & (first_events["first_addtocart_time"] >= first_events["first_view_time"])
            & (first_events["first_transaction_time"] >= first_events["first_addtocart_time"])
        ).sum()
    ]
})

strict_funnel["conversion_from_previous_step"] = strict_funnel["users"] / strict_funnel["users"].shift(1)
strict_funnel["conversion_from_first_step"] = strict_funnel["users"] / strict_funnel.loc[0, "users"]

strict_funnel

## 8. Когортный анализ удержания

Когорту определяю по месяцу первого события пользователя. Если пользователь был активен в последующие месяцы, он считается вернувшимся в соответствующем месяце жизни когорты.

Этот расчёт нужен, чтобы понять, возвращаются ли пользователи после первого взаимодействия с продуктом, и есть ли различия между когортами разных месяцев.

In [ ]:
cohort_data = con.execute("""
WITH first_seen AS (
    SELECT
        visitorid,
        DATE_TRUNC('month', MIN(event_time)) AS cohort_month
    FROM events
    GROUP BY visitorid
),
activity AS (
    SELECT DISTINCT
        visitorid,
        DATE_TRUNC('month', event_time) AS activity_month
    FROM events
),
cohort_activity AS (
    SELECT
        f.cohort_month,
        DATE_DIFF('month', f.cohort_month, a.activity_month) AS cohort_index,
        COUNT(DISTINCT a.visitorid) AS active_users
    FROM first_seen f
    JOIN activity a
        ON f.visitorid = a.visitorid
    WHERE a.activity_month >= f.cohort_month
    GROUP BY f.cohort_month, cohort_index
)
SELECT
    cohort_month,
    cohort_index,
    active_users
FROM cohort_activity
ORDER BY cohort_month, cohort_index
""").df()

cohort_data.head()

In [ ]:
cohort_sizes = cohort_data[cohort_data["cohort_index"] == 0][["cohort_month", "active_users"]]
cohort_sizes = cohort_sizes.rename(columns={"active_users": "cohort_size"})

retention = cohort_data.merge(cohort_sizes, on="cohort_month")
retention["retention_rate"] = retention["active_users"] / retention["cohort_size"]

retention_matrix = retention.pivot(
    index="cohort_month",
    columns="cohort_index",
    values="retention_rate"
)

retention_matrix

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(retention_matrix.fillna(0).values, aspect="auto")

ax.set_title("Удержание пользователей по месячным когортам")
ax.set_xlabel("Месяцев после первого события")
ax.set_ylabel("Месяц когорты")
ax.set_xticks(range(len(retention_matrix.columns)))
ax.set_xticklabels(retention_matrix.columns)
ax.set_yticks(range(len(retention_matrix.index)))
ax.set_yticklabels([d.strftime("%Y-%m") for d in retention_matrix.index])

for i in range(retention_matrix.shape[0]):
    for j in range(retention_matrix.shape[1]):
        value = retention_matrix.iloc[i, j]
        if pd.notna(value):
            ax.text(j, i, f"{value:.0%}", ha="center", va="center")

plt.tight_layout()
plt.show()

## 9. Сравнение пользовательских сегментов

В датасете нет настоящего A/B-теста, потому что пользователи не были случайно разделены на контрольную и тестовую группы. Поэтому здесь я не называю результат экспериментом.

Вместо этого я показываю близкую по логике задачу: сравниваю конверсию в покупку для пользователей, у которых первое событие произошло в будний день, и пользователей, у которых первое событие произошло в выходной. Такой пример полезен для тренировки статистической проверки различий в долях.

In [ ]:
first_user_event = (
    events
    .sort_values("event_time")
    .groupby("visitorid", as_index=False)
    .first()[["visitorid", "event_time"]]
)

first_user_event["is_weekend_first_visit"] = first_user_event["event_time"].dt.dayofweek >= 5

user_conversion = visitor_flags.merge(first_user_event[["visitorid", "is_weekend_first_visit"]], on="visitorid")
user_conversion["converted"] = user_conversion["has_transaction"]

segment_summary = (
    user_conversion
    .groupby("is_weekend_first_visit")
    .agg(
        users=("visitorid", "nunique"),
        converted_users=("converted", "sum")
    )
    .reset_index()
)

segment_summary["conversion_rate"] = segment_summary["converted_users"] / segment_summary["users"]
segment_summary

In [ ]:
from math import erf, sqrt

def two_proportion_z_test(success_a, total_a, success_b, total_b):
    p_a = success_a / total_a
    p_b = success_b / total_b
    p_pool = (success_a + success_b) / (total_a + total_b)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / total_a + 1 / total_b))
    z = (p_a - p_b) / se
    p_value = 2 * (1 - 0.5 * (1 + erf(abs(z) / sqrt(2))))
    return z, p_value

weekday = segment_summary[segment_summary["is_weekend_first_visit"] == False].iloc[0]
weekend = segment_summary[segment_summary["is_weekend_first_visit"] == True].iloc[0]

z, p_value = two_proportion_z_test(
    success_a=weekday["converted_users"],
    total_a=weekday["users"],
    success_b=weekend["converted_users"],
    total_b=weekend["users"]
)

pd.DataFrame({
    "comparison": ["первый_визит_в_будни_против_выходных"],
    "z_score": [z],
    "p_value": [p_value]
})

## 10. Итоговые продуктовые выводы

После выполнения расчётов здесь можно кратко зафиксировать основные выводы:

1. сколько пользователей дошло до каждого этапа воронки;
2. на каком переходе наблюдается самый заметный провал;
3. как быстро снижается активность пользователей по когортам;
4. отличается ли конверсия между выбранными сегментами;
5. какие продуктовые гипотезы можно предложить на основе данных.

Главная цель этого раздела — не просто вывести числа, а связать их с возможными решениями для продукта.

In [ ]:
summary = {
    "total_users": int(events["visitorid"].nunique()),
    "view_users": int(funnel.loc[funnel["step"] == "view", "users"].iloc[0]),
    "addtocart_users": int(funnel.loc[funnel["step"] == "addtocart", "users"].iloc[0]),
    "transaction_users": int(funnel.loc[funnel["step"] == "transaction", "users"].iloc[0]),
    "view_to_cart_conversion": float(funnel.loc[funnel["step"] == "addtocart", "conversion_from_previous_step"].iloc[0]),
    "cart_to_purchase_conversion": float(funnel.loc[funnel["step"] == "transaction", "conversion_from_previous_step"].iloc[0]),
}

summary

Пример итоговой интерпретации:

По воронке можно определить этап, на котором теряется основная часть пользователей. Если самый большой провал наблюдается между просмотром товара и добавлением в корзину, то в первую очередь стоит смотреть на карточку товара: понятность описания, цену, доставку, визуальное оформление и заметность кнопки добавления в корзину. Если пользователи часто добавляют товары в корзину, но не покупают, то проблема может быть уже в оформлении заказа: лишние шаги, неожиданная стоимость доставки или недостаток доверия к оплате.

Когортный анализ дополняет этот вывод: он показывает, возвращаются ли пользователи после первого взаимодействия с магазином. Если удержание быстро падает, можно проверять гипотезы про персональные рекомендации, напоминания, повторные предложения и улучшение первого пользовательского опыта.